In [2]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import scipy
import yellowbrick
import mlxtend
print("Python:", sys.version)
print("All imports work. sklearn version:", sklearn.__version__)


Python: 3.14.0 (v3.14.0:ebf955df7a8, Oct  7 2025, 08:20:14) [Clang 16.0.0 (clang-1600.0.26.6)]
All imports work. sklearn version: 1.9.1


In [3]:
import pandas as pd

df = pd.read_csv("data/raw/mental-heath-in-tech-2016_20161114.csv")
df.shape

(1433, 63)

In [4]:
df.head()

,Are you self-employed?,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health concerns and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",...,"If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?","If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?",What is your age?,What is your gender?,What country do you live in?,What US state or territory do you live in?,What country do you work in?,What US state or territory do you work in?,Which of the following best describes your work position?,Do you work remotely?
0,0,26-100,1.0,NaN,Not eligible for coverage / N/A,NaN,No,No,I don't know,Very easy,...,Not applicable to me,Not applicable to me,39,Male,United Kingdom,NaN,United Kingdom,NaN,Back-end Developer,Sometimes
1,0,6-25,1.0,NaN,No,Yes,Yes,Yes,Yes,Somewhat easy,...,Rarely,Sometimes,29,male,United States of America,Illinois,United States of America,Illinois,Back-end Developer|Front-end Developer,Never
2,0,6-25,1.0,NaN,No,NaN,No,No,I don't know,Neither easy nor difficult,...,Not applicable to me,Not applicable to me,38,Male,United Kingdom,NaN,United Kingdom,NaN,Back-end Developer,Always
3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Sometimes,Sometimes,43,male,United Kingdom,NaN,United Kingdom,NaN,Supervisor/Team Lead,Sometimes
4,0,6-25,0.0,1.0,Yes,Yes,No,No,No,Neither easy nor difficult,...,Sometimes,Sometimes,43,Female,United States of America,Illinois,United States of America,Illinois,Executive Leadership|Supervisor/Team Lead|Dev ...,Sometimes


In [5]:
df.dtypes
df.dtypes.value_counts()

str        56
int64       4
float64     3
Name: count, dtype: int64

In [6]:
df.select_dtypes(include="number").columns

Index(['Are you self-employed?',
       'Is your employer primarily a tech company/organization?',
       'Is your primary role within your company related to tech/IT?',
       'Do you have medical coverage (private insurance or state-provided) which includes treatment of  mental health issues?',
       'Do you have previous employers?',
       'Have you ever sought treatment for a mental health issue from a mental health professional?',
       'What is your age?'],
      dtype='str')

In [7]:
num = df.select_dtypes(include="number")
print(num.nunique())
print(num.isna().sum())

Are you self-employed?                                                                                                    2
Is your employer primarily a tech company/organization?                                                                   2
Is your primary role within your company related to tech/IT?                                                              2
Do you have medical coverage (private insurance or state-provided) which includes treatment of  mental health issues?     2
Do you have previous employers?                                                                                           2
Have you ever sought treatment for a mental health issue from a mental health professional?                               2
What is your age?                                                                                                        53
dtype: int64
Are you self-employed?                                                                                                 

In [8]:
selfemp = num["Are you self-employed?"]
employer_tech = num["Is your employer primarily a tech company/organization?"]
coverage = num.iloc[:, 3]

print(pd.crosstab(selfemp, employer_tech.fillna(-1)))
print(pd.crosstab(selfemp, coverage.fillna(-1)))

Is your employer primarily a tech company/organization?  -1.0   0.0   1.0
Are you self-employed?                                                   
0                                                           0   263   883
1                                                         287     0     0
Do you have medical coverage (private insurance or state-provided) which includes treatment of  mental health issues?  -1.0  \
Are you self-employed?                                                                                                        
0                                                                                                                      1146   
1                                                                                                                         0   

Do you have medical coverage (private insurance or state-provided) which includes treatment of  mental health issues?   0.0  \
Are you self-employed?                                              

In [9]:
role_tech = num.iloc[:, 2]

print(pd.crosstab(employer_tech.fillna(-1), role_tech.fillna(-1)))

Is your primary role within your company related to tech/IT?  -1.0   0.0   1.0
Is your employer primarily a tech company/organ...                            
-1.0                                                           287     0     0
 0.0                                                             0    15   248
 1.0                                                           883     0     0


In [10]:
for i, col in enumerate(df.columns):
    print(i, col)

0 Are you self-employed?
1 How many employees does your company or organization have?
2 Is your employer primarily a tech company/organization?
3 Is your primary role within your company related to tech/IT?
4 Does your employer provide mental health benefits as part of healthcare coverage?
5 Do you know the options for mental health care available under your employer-provided coverage?
6 Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?
7 Does your employer offer resources to learn more about mental health concerns and options for seeking help?
8 Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?
9 If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:
10 Do you think that discussing a mental health disorder with your employer would have negative consequences?

In [11]:
original_columns = list(df.columns)
print(len(original_columns))

names_0_17 = [
    "self_employed",
    "company_size",
    "employer_is_tech",
    "role_is_tech",
    "cur_benefits_provided",
    "cur_knows_care_options",
    "cur_employer_discussed_mh",
    "cur_resources_offered",
    "cur_anonymity_protected",
    "cur_leave_ease",
    "cur_mh_disclosure_conseq",
    "cur_ph_disclosure_conseq",
    "cur_comfort_coworkers",
    "cur_comfort_supervisor",
    "cur_mh_equals_ph",
    "cur_observed_neg_conseq",
    "mh_coverage_self_emp",
    "knows_help_resources",
]

print(len(names_0_17))
for i in range(18):
    print(i, names_0_17[i], "<-", original_columns[i][:60])

names_18_35 = [
    "reveal_to_clients",
    "reveal_clients_neg_impact",
    "reveal_to_coworkers",
    "reveal_coworkers_neg_impact",
    "productivity_affected",
    "productivity_pct_affected",
    "has_prev_employers",
    "prev_benefits_provided",
    "prev_knew_care_options",
    "prev_employer_discussed_mh",
    "prev_resources_offered",
    "prev_anonymity_protected",
    "prev_mh_disclosure_conseq",
    "prev_ph_disclosure_conseq",
    "prev_willing_coworkers",
    "prev_willing_supervisor",
    "prev_mh_equals_ph",
    "prev_observed_neg_conseq",
]

print(len(names_18_35))
for i in range(18):
    print(i + 18, names_18_35[i], "<-", original_columns[i + 18][:60])

names_36_62 = [
    "would_raise_ph_interview",
    "why_ph_interview_text",
    "would_raise_mh_interview",
    "why_mh_interview_text",
    "career_hurt_if_identified",
    "coworkers_view_negatively",
    "willing_share_friends_family",
    "observed_unsupportive_response",
    "observation_less_likely_reveal",
    "family_history_mh",
    "mh_disorder_past",
    "mh_disorder_current",
    "current_conditions_text",
    "suspected_conditions_text",
    "diagnosed_by_professional",
    "diagnosed_conditions_text",
    "sought_treatment",
    "interferes_when_treated",
    "interferes_when_not_treated",
    "age",
    "gender",
    "country_live",
    "us_state_live",
    "country_work",
    "us_state_work",
    "work_position",
    "works_remotely",
]

print(len(names_36_62))
for i in range(27):
    print(i + 36, names_36_62[i], "<-", original_columns[i + 36][:60])

new_names = names_0_17 + names_18_35 + names_36_62

df.columns = new_names

print(df.shape)
print(list(df.columns[:5]))


63
18
0 self_employed <- Are you self-employed?
1 company_size <- How many employees does your company or organization have?
2 employer_is_tech <- Is your employer primarily a tech company/organization?
3 role_is_tech <- Is your primary role within your company related to tech/IT?
4 cur_benefits_provided <- Does your employer provide mental health benefits as part of
5 cur_knows_care_options <- Do you know the options for mental health care available und
6 cur_employer_discussed_mh <- Has your employer ever formally discussed mental health (for
7 cur_resources_offered <- Does your employer offer resources to learn more about menta
8 cur_anonymity_protected <- Is your anonymity protected if you choose to take advantage 
9 cur_leave_ease <- If a mental health issue prompted you to request a medical l
10 cur_mh_disclosure_conseq <- Do you think that discussing a mental health disorder with y
11 cur_ph_disclosure_conseq <- Do you think that discussing a physical health issue with yo
12 cur

In [12]:
column_map = pd.DataFrame({
    "position": range(len(new_names)),
    "short_name": new_names,
    "original_question": original_columns,
})

print(column_map.shape)
column_map.head()

column_map.to_csv("data/column_map.csv", index=False)

check = pd.read_csv("data/column_map.csv")

(63, 3)


In [13]:
for col in ["company_size", "cur_benefits_provided", "cur_leave_ease"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

company_size
company_size
26-100            292
NaN               287
More than 1000    256
100-500           248
6-25              210
500-1000           80
1-5                60
Name: count, dtype: int64

cur_benefits_provided
cur_benefits_provided
Yes                                531
I don't know                       319
NaN                                287
No                                 213
Not eligible for coverage / N/A     83
Name: count, dtype: int64

cur_leave_ease
cur_leave_ease
NaN                           287
Somewhat easy                 281
Very easy                     220
Somewhat difficult            199
Neither easy nor difficult    178
I don't know                  150
Very difficult                118
Name: count, dtype: int64



In [14]:
scale_levels = {
    "self_employed": "binary",
    "company_size": "ordinal",
    "employer_is_tech": "binary",
    "role_is_tech": "binary",
    "cur_benefits_provided": "nominal",
    "cur_leave_ease": "ordinal",
}

print(len(scale_levels))
print(scale_levels["cur_leave_ease"])

6
ordinal


In [15]:
for col in ["cur_knows_care_options", "cur_employer_discussed_mh",
            "cur_resources_offered", "cur_anonymity_protected"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

cur_knows_care_options
cur_knows_care_options
NaN              420
No               354
I am not sure    352
Yes              307
Name: count, dtype: int64

cur_employer_discussed_mh
cur_employer_discussed_mh
No              813
NaN             287
Yes             230
I don't know    103
Name: count, dtype: int64

cur_resources_offered
cur_resources_offered
No              531
I don't know    320
Yes             295
NaN             287
Name: count, dtype: int64

cur_anonymity_protected
cur_anonymity_protected
I don't know    742
Yes             320
NaN             287
No               84
Name: count, dtype: int64



In [16]:
scale_levels.update({
    "cur_knows_care_options": "nominal",
    "cur_employer_discussed_mh": "nominal",
    "cur_resources_offered": "nominal",
    "cur_anonymity_protected": "nominal",
})

print(len(scale_levels))

10


In [17]:
for col in ["cur_mh_disclosure_conseq", "cur_ph_disclosure_conseq",
            "cur_comfort_coworkers", "cur_comfort_supervisor"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

cur_mh_disclosure_conseq
cur_mh_disclosure_conseq
Maybe    487
No       438
NaN      287
Yes      221
Name: count, dtype: int64

cur_ph_disclosure_conseq
cur_ph_disclosure_conseq
No       837
NaN      287
Maybe    268
Yes       41
Name: count, dtype: int64

cur_comfort_coworkers
cur_comfort_coworkers
Maybe    479
No       392
NaN      287
Yes      275
Name: count, dtype: int64

cur_comfort_supervisor
cur_comfort_supervisor
Yes      428
Maybe    382
No       336
NaN      287
Name: count, dtype: int64



In [18]:
scale_levels.update({
    "cur_mh_disclosure_conseq": "ordinal",
    "cur_ph_disclosure_conseq": "ordinal",
    "cur_comfort_coworkers": "ordinal",
    "cur_comfort_supervisor": "ordinal",
})

print(len(scale_levels))

14


In [19]:
for col in ["cur_mh_equals_ph", "cur_observed_neg_conseq", "knows_help_resources"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()


cur_mh_equals_ph
cur_mh_equals_ph
I don't know    493
Yes             350
No              303
NaN             287
Name: count, dtype: int64

cur_observed_neg_conseq
cur_observed_neg_conseq
No     1048
NaN     287
Yes      98
Name: count, dtype: int64

knows_help_resources
knows_help_resources
NaN                     1146
I know some              141
Yes, I know several       83
No, I don't know any      63
Name: count, dtype: int64



In [20]:
scale_levels.update({
    "cur_mh_equals_ph": "nominal",
    "cur_observed_neg_conseq": "binary",
    "mh_coverage_self_emp": "binary",
    "knows_help_resources": "ordinal",
})

print(len(scale_levels))


18


In [22]:
print(pd.crosstab(df["self_employed"], df["knows_help_resources"].fillna("missing")))

knows_help_resources  I know some  No, I don't know any  Yes, I know several  \
self_employed                                                                  
0                               0                     0                    0   
1                             141                    63                   83   

knows_help_resources  missing  
self_employed                  
0                        1146  
1                           0  
